新しい形状の方法を模索

案
1：円形度
2：楕円フィッティングと偏平率
3：現在の手法の改善

案1
最も直接的で簡単な方法の一つが「円形度」です。「真円」を1.0として、そこからどれだけ歪んでいるか（面積と周囲長の比率）を計算します。

In [1]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats # t検定のために
from sklearn.metrics import mean_squared_error # MSEは今回不要だが比較のため残置

# --- 1. 円形度 (案1) の分析クラス ---

class KeijoAnalyzer_Circularity:
    """
    シイタケのマスク画像から形状特徴（円形度）を分析する
    """
    def __init__(self):
        pass

    def analyze_circularity(self, img_path):
        """
        提案1：円形度 (Circularity) を計算する
        """
        mask = cv2.imread(img_path)
        if mask is None:
            print(f"⚠️ 読み込み失敗: {img_path}")
            return None

        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"⚠️ 輪郭なし: {img_path}")
            return None

        max_contour = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(max_contour)
        perimeter = cv2.arcLength(max_contour, True)

        if perimeter == 0:
            print(f"⚠️ 周囲長ゼロ: {img_path}")
            return None 

        circularity = (4 * np.pi * area) / (perimeter**2)
        circularity_deviation = 1.0 - circularity # 0に近いほど真円
        if circularity_deviation < 0:
            circularity_deviation = 0.0

        return circularity_deviation


class Keijo_folder_Circularity(KeijoAnalyzer_Circularity):
    """
    指定されたフォルダ内の全画像の円形度を計算し、CSVに保存する
    """
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1", output_csv=None):
        super().__init__()
        self.folder_path = os.path.join(base_dir, mask_dir, subfolder)
        self.output_csv = output_csv
        self.results_list = []

    def run(self):
        image_paths = glob.glob(os.path.join(self.folder_path, "*.jpg")) + \
                      glob.glob(os.path.join(self.folder_path, "*.png"))
        
        print(f"📂 フォルダ(円形度): {self.folder_path} に画像 {len(image_paths)} 枚")
        
        self.results_list = []

        for img_path in image_paths:
            file_name = os.path.basename(img_path)
            
            # 円形度(案1)のメソッドを呼び出す
            deviation = self.analyze_circularity(img_path)
            
            if deviation is not None:
                self.results_list.append((file_name, deviation))
            else:
                self.results_list.append((file_name, np.nan))
                
        return self.results_list

    def save_results(self):
        # 修正点：画像が0枚でも、必ずヘッダー付きの空CSVを保存する
        df = pd.DataFrame(self.results_list, columns=['filename', 'circularity_deviation'])
        df = df.set_index('filename')
        
        df.to_csv(self.output_csv)

        if not self.results_list:
            print(f"⚠️ 該当フォルダに画像がなかったため、空のCSVを保存しました: {self.output_csv}")
        else:
            print(f"✅ 結果を保存: {self.output_csv}")


# --- 2. t検定を実行する関数 ---

# （import os, pd, np, stats, plt などは関数の外側にある想定）

def run_t_test_circularity(csv_base_dir, categories_ab, category_c, output_graph_path=None):
    """
    円形度のCSVを読み込み、(A+B) 対 (C) のt検定を実行する。
    ★ 追加機能：(A+B) 対 (C) の箱ひげ図も保存する。
    """
    print("\n--- t検定 (円形度) 実行 ---")
    
    group_ab_data = []
    group_c_data = []
    
    # 1. ABグループのデータを読み込み・結合
    try:
        for category in categories_ab: # ["A", "B"]
            csv_path = os.path.join(csv_base_dir, f"keijo_circularity_{category}.csv")
            df = pd.read_csv(csv_path)
            group_ab_data.extend(df['circularity_deviation'].dropna().values)
        print(f"✅ ABグループ (円形度) 読み込み成功。データ数: {len(group_ab_data)}")
    except FileNotFoundError as e:
        print(f"❌ ABグループのCSV読み込み失敗: {e}")
        return
    except Exception as e:
        print(f"❌ エラー: {e}")
        return

    # 2. Cグループのデータを読み込み
    try:
        csv_path_c = os.path.join(csv_base_dir, f"keijo_circularity_{category_c}.csv")
        df_c = pd.read_csv(csv_path_c)
        group_c_data.extend(df_c['circularity_deviation'].dropna().values)
        print(f"✅ Cグループ (円形度) 読み込み成功。データ数: {len(group_c_data)}")
    except FileNotFoundError as e:
        print(f"❌ CグループのCSV読み込み失敗: {e}")
        return
    except Exception as e:
        print(f"❌ エラー: {e}")
        return

    # 3. t検定の実行 (前回修正済み)
    if len(group_ab_data) == 0 or len(group_c_data) == 0:
        print("❌ データが空のため、t検定・グラフ作成を実行できません。")
        return

    t_statistic, p_value = stats.ttest_ind(group_ab_data, group_c_data, equal_var=False)

    print("\n--- t検定結果 (円形度) ---")
    print(f"対象: (A + B)グループ vs Cグループ")
    print(f"t値 (statistic): {t_statistic:.4f}")
    print(f"p値 (p-value): {p_value}")
    # (解釈は省略)

    # --- 4. ★★★ 追加機能：(A+B) vs C の箱ひげ図を作成 ★★★
    if output_graph_path:
        try:
            all_data_for_plot = [group_ab_data, group_c_data]
            labels = ["A+B", "C"]

            plt.figure(figsize=(4, 6))
            
            bplot = plt.boxplot(all_data_for_plot, labels=labels, patch_artist=True)
            
            # 色付け
            colors = ['lightblue', 'lightgreen']
            for patch, color in zip(bplot['boxes'], colors):
                patch.set_facecolor(color)

            plt.title('Circularity Deviation: (A+B) vs C')
            plt.ylabel('Circularity Deviation (0 is Circle)')
            plt.grid(axis='y', linestyle='--', alpha=0.7)

            # 保存
            plt.savefig(output_graph_path)
            print(f"\n✅ 比較グラフを保存しました: {output_graph_path}")
            plt.close()

        except Exception as e:
            print(f"❌ 比較グラフの作成エラー: {e}")
    else:
        print("ℹ️ グラフの出力パスが指定されていません（グラフ作成スキップ）。")

# --- 3. メイン実行ブロック ---

if __name__ == "__main__":
    
    # --- 設定: 入力元と出力先 ---
    INPUT_DATA_DIR = "/home/data/1124_keijotest"
    OUTPUT_DIR = "/home/src/keijo/1116_newway"
    CATEGORIES = ["A", "B", "C"]
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # --- STEP 1: 円形度の分析とCSV保存 ---
    print("--- 分析実行 (円形度) ---")
    
    for category in CATEGORIES:
        # 出力するCSVファイル名を指定
        output_csv_path = os.path.join(OUTPUT_DIR, f"keijo_circularity_{category}.csv")
        
        analyzer = Keijo_folder_Circularity(
            base_dir=INPUT_DATA_DIR, 
            mask_dir="mask",
            subfolder=category,
            output_csv=output_csv_path 
        )
        analyzer.run()
        analyzer.save_results()

    comparison_graph_path = os.path.join(OUTPUT_DIR, "circularity_AB_vs_C_boxplot.png")
    
    run_t_test_circularity(
        csv_base_dir=OUTPUT_DIR,
        categories_ab=["A", "B"], 
        category_c="C",
        output_graph_path=comparison_graph_path # ★ 追加
    )

--- 分析実行 (円形度) ---
📂 フォルダ(円形度): /home/data/1124_keijotest/mask/A に画像 496 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_circularity_A.csv
📂 フォルダ(円形度): /home/data/1124_keijotest/mask/B に画像 302 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_circularity_B.csv
📂 フォルダ(円形度): /home/data/1124_keijotest/mask/C に画像 195 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_circularity_C.csv

--- t検定 (円形度) 実行 ---
✅ ABグループ (円形度) 読み込み成功。データ数: 798
✅ Cグループ (円形度) 読み込み成功。データ数: 195

--- t検定結果 (円形度) ---
対象: (A + B)グループ vs Cグループ
t値 (statistic): -4.4956
p値 (p-value): 1.0619885851953221e-05

✅ 比較グラフを保存しました: /home/src/keijo/1116_newway/circularity_AB_vs_C_boxplot.png


/tmp/ipykernel_1122139/2713202036.py:158: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot(all_data_for_plot, labels=labels, patch_artist=True)


案2
輪郭に最もフィットする「楕円」をあてはめ、その楕円がどれだけ細長いか（真円からズレているか）を計算します。

In [2]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats # t検定のために
from sklearn.metrics import mean_squared_error # 今回は不要

# --- 1. 偏平率 (案2) の分析クラス ---

class KeijoAnalyzer_Eccentricity:
    """
    シイタケのマスク画像から形状特徴（偏平率）を分析する
    """
    def __init__(self):
        pass

    def analyze_eccentricity(self, img_path):
        """
        提案2：偏平率 (Eccentricity) を計算する
        """
        mask = cv2.imread(img_path)
        if mask is None:
            print(f"⚠️ 読み込み失敗: {img_path}")
            return None

        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"⚠️ 輪郭なし: {img_path}")
            return None

        max_contour = max(contours, key=cv2.contourArea)

        # 輪郭の点が5点未満だと楕円フィッティングできない
        if len(max_contour) < 5:
            print(f"⚠️ 輪郭の点が少なすぎます: {img_path}")
            return None

        try:
            # 楕円をフィッティング
            (x, y), (minor_axis, major_axis), angle = cv2.fitEllipse(max_contour)
        except cv2.error as e:
            print(f"⚠️ cv2.fitEllipse エラー: {e} ({img_path})")
            return None

        if major_axis == 0:
            print(f"⚠️ 楕円の長軸がゼロです: {img_path}")
            return None # ゼロ除算を避ける

        # 偏平率を計算: sqrt(1 - (b/a)^2)
        # 0に近いほど真円、1に近いほど細長い
        eccentricity = np.sqrt(1 - (minor_axis / major_axis)**2)

        return eccentricity


class Keijo_folder_Eccentricity(KeijoAnalyzer_Eccentricity):
    """
    指定されたフォルダ内の全画像の偏平率を計算し、CSVに保存する
    """
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1", output_csv=None):
        super().__init__()
        self.folder_path = os.path.join(base_dir, mask_dir, subfolder)
        self.output_csv = output_csv
        self.results_list = []

    def run(self):
        image_paths = glob.glob(os.path.join(self.folder_path, "*.jpg")) + \
                      glob.glob(os.path.join(self.folder_path, "*.png"))
        
        print(f"📂 フォルダ(偏平率): {self.folder_path} に画像 {len(image_paths)} 枚")
        
        self.results_list = []

        for img_path in image_paths:
            file_name = os.path.basename(img_path)
            
            # 偏平率(案2)のメソッドを呼び出す
            ecc = self.analyze_eccentricity(img_path)
            
            if ecc is not None:
                self.results_list.append((file_name, ecc))
            else:
                self.results_list.append((file_name, np.nan))
                
        return self.results_list

    def save_results(self):
        # 画像が0枚でも、必ずヘッダー付きの空CSVを保存
        df = pd.DataFrame(self.results_list, columns=['filename', 'eccentricity'])
        df = df.set_index('filename')
        
        df.to_csv(self.output_csv)

        if not self.results_list:
            print(f"⚠️ 該当フォルダに画像がなかったため、空のCSVを保存しました: {self.output_csv}")
        else:
            print(f"✅ 結果を保存: {self.output_csv}")


# --- 2. グラフ作成の関数 (A, B, C比較) ---

def plot_eccentricity_matplotlib(csv_base_dir, categories, output_graph_path):
    """
    Matplotlibを使って 偏平率 を比較する箱ひげ図を作成・保存する (A, B, C 並列)
    """
    all_data = []
    labels = []
    print("\n--- グラフ作成開始 (偏平率: A vs B vs C) ---")

    for category in categories:
        csv_filename = f"keijo_eccentricity_{category}.csv" # ★ファイル名
        csv_path = os.path.join(csv_base_dir, csv_filename)
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                data = df['eccentricity'].dropna().values # ★列名
                if len(data) > 0:
                    all_data.append(data)
                    labels.append(category)
                    print(f"✅ 読み込み成功: {category} (データ数: {len(data)})")
                else:
                    print(f"⚠️ データなし (空のCSV): {category}")
            except Exception as e:
                print(f"❌ 読み込みエラー: {csv_path} ({e})")
        else:
            print(f"❌ ファイルが見つかりません: {csv_path}")

    if not all_data:
        print("❌ グラフで表示できるデータがありませんでした。(A vs B vs C)")
        return

    try:
        plt.figure(figsize=(10, 6))
        bplot = plt.boxplot(all_data, labels=labels, patch_artist=True)
        colors = ['pink', 'lightblue', 'lightgreen']
        for patch, color in zip(bplot['boxes'], colors * len(all_data)):
            patch.set_facecolor(color)
        plt.title('Shape Eccentricity by Category (0 is Circle)') # ★タイトル
        plt.ylabel('Eccentricity Value') # ★ラベル
        plt.xlabel('Category')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.savefig(output_graph_path)
        print(f"✅ グラフ (A vs B vs C) を保存しました: {output_graph_path}")
        plt.close()
    except Exception as e:
        print(f"❌ グラフ作成エラー (A vs B vs C): {e}")


# --- 3. t検定と (A+B) vs C グラフ作成の関数 (偏平率版) ---

def run_t_test_eccentricity(csv_base_dir, categories_ab, category_c, output_graph_path=None):
    """
    偏平率のCSVを読み込み、(A+B) 対 (C) のt検定と箱ひげ図保存を実行する
    """
    print("\n--- t検定 (偏平率) 実行 ---")
    
    group_ab_data = []
    group_c_data = []
    
    # 1. ABグループのデータを読み込み・結合
    try:
        for category in categories_ab: # ["A", "B"]
            csv_path = os.path.join(csv_base_dir, f"keijo_eccentricity_{category}.csv") # ★
            df = pd.read_csv(csv_path)
            group_ab_data.extend(df['eccentricity'].dropna().values) # ★
        print(f"✅ ABグループ (偏平率) 読み込み成功。データ数: {len(group_ab_data)}")
    except FileNotFoundError as e:
        print(f"❌ ABグループのCSV読み込み失敗: {e}")
        return
    except Exception as e:
        print(f"❌ エラー: {e}")
        return

    # 2. Cグループのデータを読み込み
    try:
        csv_path_c = os.path.join(csv_base_dir, f"keijo_eccentricity_{category_c}.csv") # ★
        df_c = pd.read_csv(csv_path_c)
        group_c_data.extend(df_c['eccentricity'].dropna().values) # ★
        print(f"✅ Cグループ (偏平率) 読み込み成功。データ数: {len(group_c_data)}")
    except FileNotFoundError as e:
        print(f"❌ CグループのCSV読み込み失敗: {e}")
        return
    except Exception as e:
        print(f"❌ エラー: {e}")
        return

    # 3. t検定の実行
    if len(group_ab_data) == 0 or len(group_c_data) == 0:
        print("❌ データが空のため、t検定・グラフ作成を実行できません。")
        return

    t_statistic, p_value = stats.ttest_ind(group_ab_data, group_c_data, equal_var=False)

    print("\n--- t検定結果 (偏平率) ---")
    print(f"対象: (A + B)グループ vs Cグループ")
    print(f"t値 (statistic): {t_statistic:.4f}")
    print(f"p値 (p-value): {p_value}")

    # 解釈
    print(f"t値の絶対値（{abs(t_statistic):.4f}）が大きいほど、ABとCは明確に分離できています。")
    if p_value < 0.05:
        print("p値が0.05より小さいため、(A+B)とCの間には「統計的に有意な差がある」と判断できます。")
    else:
        print("p値が0.05以上のため、統計的な差があるとは言い切れません。")


    # --- 4. (A+B) vs C の箱ひげ図を作成 ---
    if output_graph_path:
        try:
            all_data_for_plot = [group_ab_data, group_c_data]
            labels = ["A+B", "C"]

            plt.figure(figsize=(4, 6))
            bplot = plt.boxplot(all_data_for_plot, labels=labels, patch_artist=True)
            colors = ['lightblue', 'lightgreen']
            for patch, color in zip(bplot['boxes'], colors):
                patch.set_facecolor(color)
            
            plt.title('Eccentricity Value: (A+B) vs C') # ★
            plt.ylabel('Eccentricity Value (0 is Circle)') # ★
            plt.grid(axis='y', linestyle='--', alpha=0.7)

            plt.savefig(output_graph_path)
            print(f"\n✅ 比較グラフ (AB vs C) を保存しました: {output_graph_path}")
            plt.close()

        except Exception as e:
            print(f"❌ 比較グラフ (AB vs C) の作成エラー: {e}")
    else:
        print("ℹ️ グラフの出力パスが指定されていません（グラフ作成スキップ）。")


# --- 4. メイン実行ブロック ---

if __name__ == "__main__":
    
    # --- 設定: 入力元と出力先を分離 ---
    INPUT_DATA_DIR = "/home/data/1124_keijotest"
    OUTPUT_DIR = "/home/src/keijo/1116_newway"
    CATEGORIES = ["A", "B", "C"]
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # --- STEP 1: 分析の実行 (偏平率) ---
    print("--- 分析実行 (偏平率) ---")
    
    for category in CATEGORIES:
        output_csv_path = os.path.join(OUTPUT_DIR, f"keijo_eccentricity_{category}.csv") # ★
        
        analyzer = Keijo_folder_Eccentricity( # ★
            base_dir=INPUT_DATA_DIR,
            mask_dir="mask",
            subfolder=category,
            output_csv=output_csv_path
        )
        analyzer.run()
        analyzer.save_results()

    # --- STEP 2: グラフ作成 (A vs B vs C) ---
    graph_save_path_abc = os.path.join(OUTPUT_DIR, "eccentricity_comparison_A_B_C_boxplot.png") # ★
    
    plot_eccentricity_matplotlib( # ★
        csv_base_dir=OUTPUT_DIR,
        categories=CATEGORIES,
        output_graph_path=graph_save_path_abc
    )

    # --- STEP 3: t検定とグラフ作成 (AB vs C) の実行 ---
    graph_save_path_ab_vs_c = os.path.join(OUTPUT_DIR, "eccentricity_comparison_AB_vs_C_boxplot.png") # ★
    
    run_t_test_eccentricity( # ★
        csv_base_dir=OUTPUT_DIR,
        categories_ab=["A", "B"],
        category_c="C",
        output_graph_path=graph_save_path_ab_vs_c
    )

--- 分析実行 (偏平率) ---
📂 フォルダ(偏平率): /home/data/1124_keijotest/mask/A に画像 496 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_eccentricity_A.csv
📂 フォルダ(偏平率): /home/data/1124_keijotest/mask/B に画像 302 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_eccentricity_B.csv
📂 フォルダ(偏平率): /home/data/1124_keijotest/mask/C に画像 195 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_eccentricity_C.csv

--- グラフ作成開始 (偏平率: A vs B vs C) ---
✅ 読み込み成功: A (データ数: 496)
✅ 読み込み成功: B (データ数: 302)
✅ 読み込み成功: C (データ数: 195)
✅ グラフ (A vs B vs C) を保存しました: /home/src/keijo/1116_newway/eccentricity_comparison_A_B_C_boxplot.png

--- t検定 (偏平率) 実行 ---
✅ ABグループ (偏平率) 読み込み成功。データ数: 798
✅ Cグループ (偏平率) 読み込み成功。データ数: 195

--- t検定結果 (偏平率) ---
対象: (A + B)グループ vs Cグループ
t値 (statistic): -6.1118
p値 (p-value): 2.861551134079658e-09
t値の絶対値（6.1118）が大きいほど、ABとCは明確に分離できています。
p値が0.05より小さいため、(A+B)とCの間には「統計的に有意な差がある」と判断できます。

✅ 比較グラフ (AB vs C) を保存しました: /home/src/keijo/1116_newway/eccentricity_comparison_AB_vs_C_boxplot.png


/tmp/ipykernel_1122139/3798476206.py:139: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot(all_data, labels=labels, patch_artist=True)
/tmp/ipykernel_1122139/3798476206.py:219: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot(all_data_for_plot, labels=labels, patch_artist=True)


案3
現在のwarpPolarを使うアプローチを活かす場合、以下のように修正することで「輪郭の凹凸」をより正確に数値化できます。

    warpPolarの中心を最小外接円の中心に統一します。

    極座標変換した画像（角度 vs 半径）において、各角度（例: 0〜359度）で**最も外側にある輪郭の「半径」**を記録します。

    これにより、360個の半径値のベクトル（radius_vector）が得られます。

    このベクトルの標準偏差または変動係数（標準偏差 / 平均値）を計算します。

In [3]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats # t検定のために
from sklearn.metrics import mean_squared_error # 今回は不要

# --- 1. 半径の変動係数 (案3) の分析クラス ---

class KeijoAnalyzer_RadiusCV:
    """
    シイタケのマスク画像から形状特徴（半径の変動係数）を分析する
    """
    def __init__(self):
        pass

    def analyze_radius_cv(self, img_path):
        """
        提案3：半径ベクトルの変動係数 (CV) を計算する
        """
        mask = cv2.imread(img_path)
        if mask is None:
            print(f"⚠️ 読み込み失敗: {img_path}")
            return None

        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"⚠️ 輪郭なし: {img_path}")
            return None

        max_contour = max(contours, key=cv2.contourArea)
        
        # 1. 最小外接円の中心と半径を取得
        (x, y), radius = cv2.minEnclosingCircle(max_contour)
        center = (int(x), int(y))
        radius_max = int(radius) # 最大半径として使用

        if radius_max <= 0:
            print(f"⚠️ 半径ゼロ: {img_path}")
            return None

        # 2. 極座標変換 (角度360, 半径radius_max)
        # 角度の解像度 (例: 360)
        angle_resolution = 360
        # 出力形状は (半径の解像度, 角度の解像度)
        try:
            polar_img = cv2.warpPolar(
                th, 
                (angle_resolution, radius_max), 
                center, 
                radius_max, 
                cv2.WARP_POLAR_LINEAR
            )
        except cv2.error as e:
            print(f"⚠️ cv2.warpPolar エラー: {e} ({img_path})")
            return None

        # 3. 各角度における最大半径を記録
        radius_vector = np.zeros(angle_resolution)
        for angle in range(angle_resolution):
            # 該当角度の列（半径0からradius_maxまで）を取得
            col = polar_img[:, angle] 
            
            # 白(255)のピクセルのインデックス（=半径）を探す
            indices = np.where(col == 255)[0]
            
            if len(indices) > 0:
                # 最も外側の半径を記録
                radius_vector[angle] = np.max(indices)
            else:
                radius_vector[angle] = 0 # 輪郭がなかった角度

        # 4. 半径ベクトルの変動係数（標準偏差 / 平均）を計算
        mean_radius = np.mean(radius_vector)
        if mean_radius == 0:
            # ほぼあり得ないが、ゼロ除算を避ける
            return 0.0 

        std_dev = np.std(radius_vector)
        
        # 変動係数 (Coefficient of Variation)
        cv = std_dev / mean_radius
        
        return cv


class Keijo_folder_RadiusCV(KeijoAnalyzer_RadiusCV):
    """
    指定されたフォルダ内の全画像の半径CVを計算し、CSVに保存する
    """
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1", output_csv=None):
        super().__init__()
        self.folder_path = os.path.join(base_dir, mask_dir, subfolder)
        self.output_csv = output_csv
        self.results_list = []

    def run(self):
        image_paths = glob.glob(os.path.join(self.folder_path, "*.jpg")) + \
                      glob.glob(os.path.join(self.folder_path, "*.png"))
        
        print(f"📂 フォルダ(半径CV): {self.folder_path} に画像 {len(image_paths)} 枚")
        
        self.results_list = []

        for img_path in image_paths:
            file_name = os.path.basename(img_path)
            
            # 半径CV(案3)のメソッドを呼び出す
            cv = self.analyze_radius_cv(img_path)
            
            if cv is not None:
                self.results_list.append((file_name, cv))
            else:
                self.results_list.append((file_name, np.nan))
                
        return self.results_list

    def save_results(self):
        # 画像が0枚でも、必ずヘッダー付きの空CSVを保存
        df = pd.DataFrame(self.results_list, columns=['filename', 'radius_cv'])
        df = df.set_index('filename')
        
        df.to_csv(self.output_csv)

        if not self.results_list:
            print(f"⚠️ 該当フォルダに画像がなかったため、空のCSVを保存しました: {self.output_csv}")
        else:
            print(f"✅ 結果を保存: {self.output_csv}")


# --- 2. グラフ作成の関数 (A, B, C比較) ---

def plot_radius_cv_matplotlib(csv_base_dir, categories, output_graph_path):
    """
    Matplotlibを使って 半径CV を比較する箱ひげ図を作成・保存する (A, B, C 並列)
    """
    all_data = []
    labels = []
    print("\n--- グラフ作成開始 (半径CV: A vs B vs C) ---")

    for category in categories:
        csv_filename = f"keijo_radius_cv_{category}.csv" # ★
        csv_path = os.path.join(csv_base_dir, csv_filename)
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                data = df['radius_cv'].dropna().values # ★
                if len(data) > 0:
                    all_data.append(data)
                    labels.append(category)
                    print(f"✅ 読み込み成功: {category} (データ数: {len(data)})")
                else:
                    print(f"⚠️ データなし (空のCSV): {category}")
            except Exception as e:
                print(f"❌ 読み込みエラー: {csv_path} ({e})")
        else:
            print(f"❌ ファイルが見つかりません: {csv_path}")

    if not all_data:
        print("❌ グラフで表示できるデータがありませんでした。(A vs B vs C)")
        return

    try:
        plt.figure(figsize=(10, 6))
        bplot = plt.boxplot(all_data, labels=labels, patch_artist=True)
        colors = ['pink', 'lightblue', 'lightgreen']
        for patch, color in zip(bplot['boxes'], colors * len(all_data)):
            patch.set_facecolor(color)
        plt.title('Shape Radius CV by Category (0 is Circle)') # ★
        plt.ylabel('Radius CV Value (StdDev / Mean)') # ★
        plt.xlabel('Category')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.savefig(output_graph_path)
        print(f"✅ グラフ (A vs B vs C) を保存しました: {output_graph_path}")
        plt.close()
    except Exception as e:
        print(f"❌ グラフ作成エラー (A vs B vs C): {e}")


# --- 3. t検定と (A+B) vs C グラフ作成の関数 (半径CV版) ---

def run_t_test_radius_cv(csv_base_dir, categories_ab, category_c, output_graph_path=None):
    """
    半径CVのCSVを読み込み、(A+B) 対 (C) のt検定と箱ひげ図保存を実行する
    """
    print("\n--- t検定 (半径CV) 実行 ---")
    
    group_ab_data = []
    group_c_data = []
    
    # 1. ABグループのデータを読み込み・結合
    try:
        for category in categories_ab: # ["A", "B"]
            csv_path = os.path.join(csv_base_dir, f"keijo_radius_cv_{category}.csv") # ★
            df = pd.read_csv(csv_path)
            group_ab_data.extend(df['radius_cv'].dropna().values) # ★
        print(f"✅ ABグループ (半径CV) 読み込み成功。データ数: {len(group_ab_data)}")
    except FileNotFoundError as e:
        print(f"❌ ABグループのCSV読み込み失敗: {e}")
        return
    except Exception as e:
        print(f"❌ エラー: {e}")
        return

    # 2. Cグループのデータを読み込み
    try:
        csv_path_c = os.path.join(csv_base_dir, f"keijo_radius_cv_{category_c}.csv") # ★
        df_c = pd.read_csv(csv_path_c)
        group_c_data.extend(df_c['radius_cv'].dropna().values) # ★
        print(f"✅ Cグループ (半径CV) 読み込み成功。データ数: {len(group_c_data)}")
    except FileNotFoundError as e:
        print(f"❌ CグループのCSV読み込み失敗: {e}")
        return
    except Exception as e:
        print(f"❌ エラー: {e}")
        return

    # 3. t検定の実行
    if len(group_ab_data) == 0 or len(group_c_data) == 0:
        print("❌ データが空のため、t検定・グラフ作成を実行できません。")
        return

    t_statistic, p_value = stats.ttest_ind(group_ab_data, group_c_data, equal_var=False)

    print("\n--- t検定結果 (半径CV) ---")
    print(f"対象: (A + B)グループ vs Cグループ")
    print(f"t値 (statistic): {t_statistic:.4f}")
    print(f"p値 (p-value): {p_value}")

    # 解釈
    print(f"t値の絶対値（{abs(t_statistic):.4f}）が大きいほど、ABとCは明確に分離できています。")
    if p_value < 0.05:
        print("p値が0.05より小さいため、(A+B)とCの間には「統計的に有意な差がある」と判断できます。")
    else:
        print("p値が0.05以上のため、統計的な差があるとは言い切れません。")


    # --- 4. (A+B) vs C の箱ひげ図を作成 ---
    if output_graph_path:
        try:
            all_data_for_plot = [group_ab_data, group_c_data]
            labels = ["A+B", "C"]

            plt.figure(figsize=(4, 6))
            bplot = plt.boxplot(all_data_for_plot, labels=labels, patch_artist=True)
            colors = ['lightblue', 'lightgreen']
            for patch, color in zip(bplot['boxes'], colors):
                patch.set_facecolor(color)
            
            plt.title('Radius CV Value: (A+B) vs C') # ★
            plt.ylabel('Radius CV Value (0 is Circle)') # ★
            plt.grid(axis='y', linestyle='--', alpha=0.7)

            plt.savefig(output_graph_path)
            print(f"\n✅ 比較グラフ (AB vs C) を保存しました: {output_graph_path}")
            plt.close()

        except Exception as e:
            print(f"❌ 比較グラフ (AB vs C) の作成エラー: {e}")
    else:
        print("ℹ️ グラフの出力パスが指定されていません（グラフ作成スキップ）。")


# --- 4. メイン実行ブロック ---

if __name__ == "__main__":
    
    # --- 設定: 入力元と出力先を分離 ---
    INPUT_DATA_DIR = "/home/data/1124_keijotest"
    OUTPUT_DIR = "/home/src/keijo/1116_newway"
    CATEGORIES = ["A", "B", "C"]
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # --- STEP 1: 分析の実行 (半径CV) ---
    print("--- 分析実行 (半径CV) ---")
    
    for category in CATEGORIES:
        output_csv_path = os.path.join(OUTPUT_DIR, f"keijo_radius_cv_{category}.csv") # ★
        
        analyzer = Keijo_folder_RadiusCV( # ★
            base_dir=INPUT_DATA_DIR,
            mask_dir="mask",
            subfolder=category,
            output_csv=output_csv_path
        )
        analyzer.run()
        analyzer.save_results()

    # --- STEP 2: グラフ作成 (A vs B vs C) ---
    graph_save_path_abc = os.path.join(OUTPUT_DIR, "radius_cv_comparison_A_B_C_boxplot.png") # ★
    
    plot_radius_cv_matplotlib( # ★
        csv_base_dir=OUTPUT_DIR,
        categories=CATEGORIES,
        output_graph_path=graph_save_path_abc
    )

    # --- STEP 3: t検定とグラフ作成 (AB vs C) の実行 ---
    graph_save_path_ab_vs_c = os.path.join(OUTPUT_DIR, "radius_cv_comparison_AB_vs_C_boxplot.png") # ★
    
    run_t_test_radius_cv( # ★
        csv_base_dir=OUTPUT_DIR,
        categories_ab=["A", "B"],
        category_c="C",
        output_graph_path=graph_save_path_ab_vs_c
    )

--- 分析実行 (半径CV) ---
📂 フォルダ(半径CV): /home/data/1124_keijotest/mask/A に画像 496 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_radius_cv_A.csv
📂 フォルダ(半径CV): /home/data/1124_keijotest/mask/B に画像 302 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_radius_cv_B.csv
📂 フォルダ(半径CV): /home/data/1124_keijotest/mask/C に画像 195 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_radius_cv_C.csv

--- グラフ作成開始 (半径CV: A vs B vs C) ---
✅ 読み込み成功: A (データ数: 496)
✅ 読み込み成功: B (データ数: 302)
✅ 読み込み成功: C (データ数: 195)
✅ グラフ (A vs B vs C) を保存しました: /home/src/keijo/1116_newway/radius_cv_comparison_A_B_C_boxplot.png

--- t検定 (半径CV) 実行 ---
✅ ABグループ (半径CV) 読み込み成功。データ数: 798
✅ Cグループ (半径CV) 読み込み成功。データ数: 195

--- t検定結果 (半径CV) ---
対象: (A + B)グループ vs Cグループ
t値 (statistic): 0.5787
p値 (p-value): 0.5632221004649947
t値の絶対値（0.5787）が大きいほど、ABとCは明確に分離できています。
p値が0.05以上のため、統計的な差があるとは言い切れません。

✅ 比較グラフ (AB vs C) を保存しました: /home/src/keijo/1116_newway/radius_cv_comparison_AB_vs_C_boxplot.png


/tmp/ipykernel_1122139/830890142.py:170: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot(all_data, labels=labels, patch_artist=True)
/tmp/ipykernel_1122139/830890142.py:250: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot(all_data_for_plot, labels=labels, patch_artist=True)


元のグラフ出すよう

In [4]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from scipy import stats # ★ t検定のために追加

# --- 1. 分析クラスの定義 (元のMSEロジックを使用) ---
# (ここは変更ありません)

class KeijoAnalyzer:
    """
    シイタケのマスク画像から形状特徴（元のMSE）を分析する
    """
    def __init__(self):
        pass

    def analyze_image(self, img_path):
        mask = cv2.imread(img_path)
        if mask is None:
            print(f"⚠️ 読み込み失敗: {img_path}")
            return None, None

        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"⚠️ 輪郭なし: {img_path}")
            return None, None

        max_contour = max(contours, key=cv2.contourArea)
        (x, y), radius = cv2.minEnclosingCircle(max_contour)
        radius = int(radius)

        M = cv2.moments(max_contour)
        cX, cY = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])) if M["m00"] != 0 else (0, 0)
        center = (cX, cY)

        h, w = gray.shape
        flags = cv2.INTER_CUBIC + cv2.WARP_FILL_OUTLIERS + cv2.WARP_POLAR_LINEAR
        
        if radius <= 0:
            print(f"⚠️ 半径ゼロ: {img_path}")
            return None, None
            
        linear_polar = cv2.warpPolar(gray, (w, h), center, radius, flags)

        black_pixel_count = np.sum(linear_polar == 0, axis=1)
        
        if black_pixel_count.size == 0:
            print(f"⚠️ warpPolar結果なし: {img_path}")
            return None, None

        y_pseudo = np.zeros_like(black_pixel_count)
        mse = mean_squared_error(y_pseudo, black_pixel_count)

        return black_pixel_count, mse


class Keijo_file(KeijoAnalyzer):
    # (変更なし)
    def __init__(self, img_path):
        super().__init__()
        self.img_path = img_path

    def run(self):
        vec, mse = self.analyze_image(self.img_path)
        return mse


class Keijo_folder(KeijoAnalyzer):
    # (変更なし)
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1", output_csv=None):
        super().__init__()
        self.folder_path = os.path.join(base_dir, mask_dir, subfolder)
        self.output_csv = output_csv
        self.results_list = []

    def run(self):
        image_paths = glob.glob(os.path.join(self.folder_path, "*.jpg")) + \
                      glob.glob(os.path.join(self.folder_path, "*.png"))
        print(f"📂 フォルダ: {self.folder_path} に画像 {len(image_paths)} 枚")
        self.results_list = []
        for img_path in image_paths:
            file_name = os.path.basename(img_path)
            vec, mse = self.analyze_image(img_path)
            if mse is not None:
                self.results_list.append((file_name, mse))
            else:
                self.results_list.append((file_name, np.nan))
        return self.results_list

    def save_results(self):
        # (変更なし - 0枚でも空CSVを保存)
        df = pd.DataFrame(self.results_list, columns=['filename', 'mse'])
        df = df.set_index('filename')
        df.to_csv(self.output_csv)
        if not self.results_list:
            print(f"⚠️ 該当フォルダに画像がなかったため、空のCSVを保存しました: {self.output_csv}")
        else:
            print(f"✅ 結果を保存: {self.output_csv}")


# --- 2. グラフ作成の関数 (A, B, C比較) ---
# (変更なし)

def plot_mse_matplotlib(csv_base_dir, categories, output_graph_path):
    """
    Matplotlibを使って MSE を比較する箱ひげ図を作成・保存する (A, B, C 並列)
    """
    all_data = []
    labels = []
    print("\n--- グラフ作成開始 (MSE: A vs B vs C) ---")

    for category in categories:
        csv_filename = f"keijo_mse_{category}.csv"
        csv_path = os.path.join(csv_base_dir, csv_filename)
        if os.path.exists(csv_path):
            try:
                df = pd.read_csv(csv_path)
                data = df['mse'].dropna().values
                if len(data) > 0:
                    all_data.append(data)
                    labels.append(category)
                    print(f"✅ 読み込み成功: {category} (データ数: {len(data)})")
                else:
                    print(f"⚠️ データなし (空のCSV): {category}")
            except Exception as e:
                print(f"❌ 読み込みエラー: {csv_path} ({e})")
        else:
            print(f"❌ ファイルが見つかりません: {csv_path}")

    if not all_data:
        print("❌ グラフで表示できるデータがありませんでした。(A vs B vs C)")
        return

    try:
        plt.figure(figsize=(10, 6))
        bplot = plt.boxplot(all_data, labels=labels, patch_artist=True)
        colors = ['pink', 'lightblue', 'lightgreen']
        for patch, color in zip(bplot['boxes'], colors * len(all_data)):
            patch.set_facecolor(color)
        plt.title('Shape MSE by Category (Original Method)')
        plt.ylabel('MSE Value')
        plt.xlabel('Category')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.savefig(output_graph_path)
        print(f"✅ グラフ (A vs B vs C) を保存しました: {output_graph_path}")
        plt.close()
    except Exception as e:
        print(f"❌ グラフ作成エラー (A vs B vs C): {e}")


# --- 3. ★★★ t検定と (A+B) vs C グラフ作成の関数 (MSE版) ★★★ ---
# (円形度版をコピーし、列名などを 'mse' に変更)

def run_t_test_mse(csv_base_dir, categories_ab, category_c, output_graph_path=None):
    """
    MSEのCSVを読み込み、(A+B) 対 (C) のt検定と箱ひげ図保存を実行する
    """
    print("\n--- t検定 (MSE) 実行 ---")
    
    group_ab_data = []
    group_c_data = []
    
    # 1. ABグループのデータを読み込み・結合
    try:
        for category in categories_ab: # ["A", "B"]
            csv_path = os.path.join(csv_base_dir, f"keijo_mse_{category}.csv")
            df = pd.read_csv(csv_path)
            # ★ 読み込む列を 'mse' に変更
            group_ab_data.extend(df['mse'].dropna().values)
        print(f"✅ ABグループ (MSE) 読み込み成功。データ数: {len(group_ab_data)}")
    except FileNotFoundError as e:
        print(f"❌ ABグループのCSV読み込み失敗: {e}")
        return
    except Exception as e:
        print(f"❌ エラー: {e}")
        return

    # 2. Cグループのデータを読み込み
    try:
        csv_path_c = os.path.join(csv_base_dir, f"keijo_mse_{category_c}.csv")
        df_c = pd.read_csv(csv_path_c)
        # ★ 読み込む列を 'mse' に変更
        group_c_data.extend(df_c['mse'].dropna().values)
        print(f"✅ Cグループ (MSE) 読み込み成功。データ数: {len(group_c_data)}")
    except FileNotFoundError as e:
        print(f"❌ CグループのCSV読み込み失敗: {e}")
        return
    except Exception as e:
        print(f"❌ エラー: {e}")
        return

    # 3. t検定の実行
    if len(group_ab_data) == 0 or len(group_c_data) == 0:
        print("❌ データが空のため、t検定・グラフ作成を実行できません。")
        return

    t_statistic, p_value = stats.ttest_ind(group_ab_data, group_c_data, equal_var=False)

    print("\n--- t検定結果 (MSE) ---")
    print(f"対象: (A + B)グループ vs Cグループ")
    print(f"t値 (statistic): {t_statistic:.4f}")
    print(f"p値 (p-value): {p_value}")

    # 解釈
    print(f"t値の絶対値（{abs(t_statistic):.4f}）が大きいほど、ABとCは明確に分離できています。")
    if p_value < 0.05:
        print("p値が0.05より小さいため、(A+B)とCの間には「統計的に有意な差がある」と判断できます。")
    else:
        print("p値が0.05以上のため、統計的な差があるとは言い切れません。")


    # --- 4. (A+B) vs C の箱ひげ図を作成 ---
    if output_graph_path:
        try:
            all_data_for_plot = [group_ab_data, group_c_data]
            labels = ["A+B", "C"]

            plt.figure(figsize=(4, 6))
            bplot = plt.boxplot(all_data_for_plot, labels=labels, patch_artist=True)
            colors = ['lightblue', 'lightgreen']
            for patch, color in zip(bplot['boxes'], colors):
                patch.set_facecolor(color)
            
            # ★ タイトルとラベルを 'MSE' 用に変更
            plt.title('MSE Value: (A+B) vs C')
            plt.ylabel('MSE Value')
            plt.grid(axis='y', linestyle='--', alpha=0.7)

            plt.savefig(output_graph_path)
            print(f"\n✅ 比較グラフ (AB vs C) を保存しました: {output_graph_path}")
            plt.close()

        except Exception as e:
            print(f"❌ 比較グラフ (AB vs C) の作成エラー: {e}")
    else:
        print("ℹ️ グラフの出力パスが指定されていません（グラフ作成スキップ）。")


# --- 4. メイン実行ブロック ---

if __name__ == "__main__":
    
    # --- 設定: 入力元と出力先を分離 ---
    INPUT_DATA_DIR = "/home/data/1124_keijotest"
    OUTPUT_DIR = "/home/src/keijo/1116_newway"
    CATEGORIES = ["A", "B", "C"]
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # --- STEP 1: 分析の実行 (MSE) ---
    print("--- 分析実行 (MSE) ---")
    
    for category in CATEGORIES:
        output_csv_path = os.path.join(OUTPUT_DIR, f"keijo_mse_{category}.csv")
        
        analyzer = Keijo_folder(
            base_dir=INPUT_DATA_DIR,
            mask_dir="mask",
            subfolder=category,
            output_csv=output_csv_path
        )
        analyzer.run()
        analyzer.save_results()

    # --- STEP 2: グラフ作成 (A vs B vs C) ---
    graph_save_path_abc = os.path.join(OUTPUT_DIR, "mse_comparison_A_B_C_boxplot.png")
    
    plot_mse_matplotlib(
        csv_base_dir=OUTPUT_DIR,
        categories=CATEGORIES,
        output_graph_path=graph_save_path_abc
    )

    # --- STEP 3: ★★★ t検定とグラフ作成 (AB vs C) の実行 ★★★ ---
    graph_save_path_ab_vs_c = os.path.join(OUTPUT_DIR, "mse_comparison_AB_vs_C_boxplot.png")
    
    run_t_test_mse(
        csv_base_dir=OUTPUT_DIR,
        categories_ab=["A", "B"],
        category_c="C",
        output_graph_path=graph_save_path_ab_vs_c
    )

--- 分析実行 (MSE) ---
📂 フォルダ: /home/data/1124_keijotest/mask/A に画像 496 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_mse_A.csv
📂 フォルダ: /home/data/1124_keijotest/mask/B に画像 302 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_mse_B.csv
📂 フォルダ: /home/data/1124_keijotest/mask/C に画像 195 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_mse_C.csv

--- グラフ作成開始 (MSE: A vs B vs C) ---
✅ 読み込み成功: A (データ数: 496)
✅ 読み込み成功: B (データ数: 302)
✅ 読み込み成功: C (データ数: 195)
✅ グラフ (A vs B vs C) を保存しました: /home/src/keijo/1116_newway/mse_comparison_A_B_C_boxplot.png

--- t検定 (MSE) 実行 ---
✅ ABグループ (MSE) 読み込み成功。データ数: 798
✅ Cグループ (MSE) 読み込み成功。データ数: 195

--- t検定結果 (MSE) ---
対象: (A + B)グループ vs Cグループ
t値 (statistic): -5.8381
p値 (p-value): 1.4850267615225227e-08
t値の絶対値（5.8381）が大きいほど、ABとCは明確に分離できています。
p値が0.05より小さいため、(A+B)とCの間には「統計的に有意な差がある」と判断できます。

✅ 比較グラフ (AB vs C) を保存しました: /home/src/keijo/1116_newway/mse_comparison_AB_vs_C_boxplot.png


/tmp/ipykernel_1122139/2160328651.py:142: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot(all_data, labels=labels, patch_artist=True)
/tmp/ipykernel_1122139/2160328651.py:225: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot(all_data_for_plot, labels=labels, patch_artist=True)


In [ ]:
import os
import glob
import cv2
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import pandas as pd

class KeijoAnalyzer:
    def __init__(self):
        self.results = {}
        self.data_vectors = {}

    def analyze_image(self, img_path):
        mask = cv2.imread(img_path)
        if mask is None:
            print(f"⚠️ 読み込み失敗: {img_path}")
            return None, None

        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"⚠️ 輪郭なし: {img_path}")
            return None, None

        max_contour = max(contours, key=cv2.contourArea)
        (x, y), radius = cv2.minEnclosingCircle(max_contour)
        radius = int(radius)

        M = cv2.moments(max_contour)
        cX, cY = (int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])) if M["m00"] != 0 else (0, 0)
        center = (cX, cY)

        h, w = gray.shape
        flags = cv2.INTER_CUBIC + cv2.WARP_FILL_OUTLIERS + cv2.WARP_POLAR_LINEAR
        linear_polar = cv2.warpPolar(gray, (w, h), center, radius, flags)

        black_pixel_count = np.sum(linear_polar == 0, axis=1)

        # 評価指標の計算
        y_pseudo = np.zeros_like(black_pixel_count)
        mse = mean_squared_error(y_pseudo, black_pixel_count)

        return black_pixel_count, mse


class Keijo_file(KeijoAnalyzer):
    def __init__(self, img_path):
        super().__init__()
        self.img_path = img_path

    def run(self):
        _, mse = self.analyze_image(self.img_path)
        return mse  # MSE のみ返す


class Keijo_folder(KeijoAnalyzer):
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1", output_csv=None):
        super().__init__()
        self.base_dir = base_dir
        self.mask_dir = mask_dir
        self.subfolder = subfolder
        self.folder_path = os.path.join(base_dir, mask_dir, subfolder)
        self.output_csv = output_csv or os.path.join(base_dir, "keijo_mse.csv")

    def run(self):
        image_paths = glob.glob(os.path.join(self.folder_path, "*.jpg")) + glob.glob(os.path.join(self.folder_path, "*.png"))
        print(f"📂 フォルダ: {self.folder_path} に画像 {len(image_paths)} 枚")
        mse_results = []

        for img_path in image_paths:
            file_name = os.path.basename(img_path)
            vec, mse = self.analyze_image(img_path)
            mse_results.append((file_name, mse))
        return mse_results

    def save_results(self):
        df = pd.DataFrame.from_dict(self.results, orient='index')
        df.index.name = 'filename'
        df.to_csv(self.output_csv)
        print(f"✅ 結果を保存: {self.output_csv}")


# import keijo
# analyzer = keijo.Keijo_file("/home/data/maesyori_img/mask/collage_1/mask_1_1.jpg")
# result = analyzer.run()
# print(result)
# import keijo
# analyzer = keijo.Keijo_folder(base_dir="/home/data/maesyori_img")
# result = analyzer.run()
# print(result)